This file contains my solution to a task from [Data Science Simulator](https://karpov.courses/simulator-ds), a hands-on training course in data analysis and machine learning by karpov.courses.

This file is shared with the permission of the course authors, and in a reduced form: the original problem statements and datasets are omitted. What's kept is my own code, reasoning, and commentary.

### Overview

This is a sentiment analysis task: figure out from a review left on a food delivery service whether the customer liked their order or not.

#### General approach

I solve it via fine-tuning a BERT model. Fine-tuning here means taking the pretrained [CLS] token and training a logistic regression on top of it.

### Step 1. Tokenization

To fine-tune the model the text needs to be tokenized. I use the DistilBERT model and its corresponding tokenizer — a lightweight and fast model that fits the constraints of this task.

Because of memory limits I tokenize in batches. I need to write a utility that loads the i-th batch from the downloaded dataset and tokenizes it.

The dataset is an export of customer reviews. It contains customer ratings from 1 to 5. The sentiment is derived from the rating. In the class, `labels` returns -1, 0, or +1 depending on the review's sentiment.

The `DataLoader` class defines the `__iter__` iterator, which lets you iterate over batches. During iteration each batch gets tokenized. The tokenizer outputs numeric indices, each corresponding to a token in the model.

In [ ]:
from dataclasses import dataclass
from transformers import PreTrainedTokenizer
from typing import List, Generator, Tuple

@dataclass
class DataLoader:
    path: str
    tokenizer: PreTrainedTokenizer
    batch_size: int = 512 #Number of batches that are proccessed at once
    max_length: int = 128 #Maxium length of a review
    
    def __iter__(self) -> Generator[List[List[int]], None, None]:
        """Iterate over batches"""
        for i in range(len(self)):
            yield self.batch_tokenized(i)
            
    def __len__(self) -> int:
        """Number of batches"""
        with open(self.path) as f:
            n_rows = sum(1 for _ in f)
        n_rows -= 1  # Skip header
        return -(-n_rows//self.batch_size)
    
    def tokenize(self, batch: List[str]) -> List[List[int]]:
        """Tokenize list of texts"""
        tokenized_batch = [
            self.tokenizer.encode(
                x,
                add_special_tokens=True,
                max_length=self.max_length,
                truncation=True
            )
            for x in batch
        ]
        return tokenized_batch
    
    def batch_loaded(self, i: int) -> Tuple[List[str], List[int]]:
        """Return loaded i-th batch of data (text, label)"""
        index_start = i * self.batch_size
        index_end = (i + 1) * self.batch_size

        texts = []
        labels = []
        with open(self.path) as f:
            _ = next(f)  # Skip header
            for j, line in enumerate(f):
                if index_start <= j < index_end:
                    fields = line.split(",", 4)

                    sentiment = fields[3]
                    if sentiment == "positive":
                        label = 1
                    elif sentiment == "negative":
                        label = -1
                    else:
                        label = 0

                    texts.append(fields[4].strip())
                    labels.append(label)

                if j >= index_end:
                    break

        return texts, labels
    
    def batch_tokenized(self, i: int) -> Tuple[List[List[int]], List[int]]:
        """Return tokenized i-th batch of data"""
        texts, labels = self.batch_loaded(i)
        tokens = self.tokenize(texts)
        return tokens, labels

#### The hidden cell below contains a solution that loads the whole dataset into memory right away

In [ ]:
from dataclasses import dataclass
from transformers import PreTrainedTokenizer
from typing import List, Generator, Tuple

@dataclass
class DataLoader:
    path: str
    tokenizer: PreTrainedTokenizer
    batch_size: int = 512 #Number of batches that are proccessed at once
    max_length: int = 128 #Maxium length of a review
    
    
    def __post_init__(self) -> None:
        self.reviews = []
        with open(self.path, "r") as f:
            for line in f:
                self.reviews.append(line.strip().split(",", 4))
        self.reviews = self.reviews[1:]
    
    
    def __iter__(self) -> Generator[List[List[int]], None, None]:
        """Iterate over batches"""
        for i in range(len(self)):
            yield self.batch_tokenized(i)
            
    def __len__(self) -> int:
        """Number of batches"""
        length = len(self.reviews)
        return -(-length//self.batch_size)
    
    def tokenize(self, batch: List[str]) -> List[List[int]]:
        """Tokenize list of texts"""
        tokenized_batch = [
            self.tokenizer.encode(
                x,
                add_special_tokens=True,
                max_length=self.max_length,
                truncation=True
            )
            for x in batch
        ]
        return tokenized_batch
    
    def batch_loaded(self, i: int) -> Tuple[List[str], List[int]]:
        """Return loaded i-th batch of data (text, label)"""
        start = i * self.batch_size
        end = (i + 1) * self.batch_size if i != len(self) - 1 else len(self.reviews)
        batch = self.reviews[start:end]
        
        labels = []
        texts = []
        for r in batch:
            if r[3] == "positive":
                labels.append(1)
            elif r[3] == "neutral":
                labels.append(0)
            else:
                labels.append(-1)
            texts.append(r[4])
        return texts, labels
    
    def batch_tokenized(self, i: int) -> Tuple[List[List[int]], List[int]]:
        """Return tokenized i-th batch of data"""
        texts, labels = self.batch_loaded(i)
        tokens = self.tokenize(texts)
        return tokens, labels

### Step 2. Padding

The model requires matrices of equal size to work with. In our batches the tokenized texts have different lengths. Two types of padding need to be added:
- Normal Padding — take the maximum sentence length across the whole dataset and pad every sentence up to that length (in this task I simply use the maximum allowed length set when the class is defined)
- Dynamic Padding — take the maximum sentence length within the current batch and pad every sentence in the batch up to that length. Dynamic padding is more efficient computationally.

After adding padding I need to implement a function that builds the Attention Mask. This function creates a list of 1s and 0s, where 1 corresponds to real dataset tokens and 0 to padding. This is needed so the model ignores the padding when computing the final result (since regardless of the padding length we should get the same result).

I also need to implement the `review_embedding` function, which builds the embedding for a single batch.

In [ ]:
from dataclasses import dataclass
from typing import List, Generator, Tuple

from transformers import PreTrainedTokenizer
import torch

@dataclass
class DataLoader:
    path: str
    tokenizer: PreTrainedTokenizer
    batch_size: int = 512 #Number of batches that are proccessed at once
    max_length: int = 128 #Maxium length of a review
    padding: str = None #[None, 'max_length', 'batch'] are allowed

    def __iter__(self) -> Generator[List[List[int]], None, None]:
        """Iterate over batches"""
        for i in range(len(self)):
            yield self.batch_tokenized(i)

    def __len__(self) -> int:
        """Number of batches"""
        with open(self.path) as f:
            n_rows = sum(1 for _ in f)
        n_rows -= 1  # Skip header
        return -(-n_rows//self.batch_size)

    def tokenize(self, batch: List[str]) -> List[List[int]]:
        """Tokenize list of texts"""
        tokenized_batch = [
            self.tokenizer.encode(
                x,
                add_special_tokens=True,
                max_length=self.max_length,
                truncation=True
            )
            for x in batch
        ]

        #Process the padding
        if self.padding == "max_length":
            max_len = self.max_length
        elif self.padding == "batch":
            max_len = max(len(t) for t in tokenized_batch)
        else:
            return tokenized_batch

        for t in tokenized_batch:
            if len(t) < max_len:
                t.extend([0] * (max_len - len(t)))

        return tokenized_batch

    def batch_loaded(self, i: int) -> Tuple[List[str], List[int]]:
        """Return loaded i-th batch of data (text, label)"""
        index_start = i * self.batch_size
        index_end = (i + 1) * self.batch_size

        texts = []
        labels = []
        with open(self.path) as f:
            _ = next(f)  # Skip header
            for j, line in enumerate(f):
                if index_start <= j < index_end:
                    fields = line.split(",", 4)

                    sentiment = fields[3]
                    if sentiment == "positive":
                        label = 1
                    elif sentiment == "negative":
                        label = -1
                    else:
                        label = 0

                    texts.append(fields[4].strip())
                    labels.append(label)

                if j >= index_end:
                    break

        return texts, labels

    def batch_tokenized(self, i: int) -> Tuple[List[List[int]], List[int]]:
        """Return tokenized i-th batch of data"""
        texts, labels = self.batch_loaded(i)
        tokens = self.tokenize(texts)
        return tokens, labels

def attention_mask(padded: List[List[int]]) -> List[List[int]]:
    """Create attention mask so the model knows location of padded tokens"""
    masked_batch = []
    for p in padded:
        masked_review = [1 if i != 0 else 0 for i in p]
        masked_batch.append(masked_review)
    return masked_batch

def review_embedding(tokens: List[List[int]], model) -> List[List[float]]:
    """Return embedding for batch of tokenized texts"""

    #Create attention mask
    mask = attention_mask(tokens)

    #Calculate embeddings
    tokens = torch.tensor(tokens)
    mask = torch.tensor(mask)
    with torch.no_grad():
        last_hidden_state = model(tokens, attention_mask=mask)

    #Return embeddings for the [CLS]-tokens that contain reviews context
    #[:,0,:] means that I take all sentences from the batch,
    #first token that corresponds to the [CLS]-token and all dimensions
    #of the token
    features = last_hidden_state[0][:,0,:].tolist()

    return features


In [ ]:
from transformers import DistilBertModel, DistilBertTokenizer
#Uncased means that the model doesn't make difference between "Bert" and "bert"
MODEL_NAME = 'distilbert-base-uncased'

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
bert = DistilBertModel.from_pretrained(MODEL_NAME)

### Step 3. Sentiment classification via logistic regression

The final stage — training a model on top of the obtained embeddings. For this task (3-class classification) I use plain logistic regression from scikit-learn.

Training uses 5-fold cross-validation (without shuffling). I need to write an `evaluate` function that takes embeddings and labels as input and returns the Cross-Entropy Loss for each fold.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedKFold

log_reg_model = LogisticRegression(max_iter=500)

def evaluate(model, embeddings: List[List[float]], labels: List[int], cv: int = 5) -> List[float]:
    """Apply CV and calculate cross-entropy scores"""
    cv = StratifiedKFold(n_splits=cv, shuffle=False)
    cv_results = cross_validate(model, embeddings, labels, scoring='neg_log_loss', cv=cv)
    scores = -cv_results['test_score']
    return scores